# DESlab en Python 3 / Google Colab

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nebisman/deslab-py3/blob/main/notebooks/DESlab_Colab_class.ipynb)

DESlab es una herramienta para el análisis y la síntesis de **sistemas a eventos discretos** modelados como autómatas
(L. Bermeo). Esta versión funciona con Python 3 y las versiones actuales de `networkx` y `graphviz`.
Cada llamada a `draw(...)` muestra los autómatas **directamente en el notebook**.

Repositorio: https://github.com/nebisman/deslab-py3



In [ ]:
#@title Instalación de DESlab desde GitHub (ejecute esta celda primero)
import shutil, subprocess

# programas de graphviz (Colab normalmente ya los trae instalados)
if shutil.which("dot") is None:
    subprocess.run("apt-get -qq install -y graphviz > /dev/null", shell=True)

%pip install -q git+https://github.com/nebisman/deslab-py3.git

In [ ]:
from deslab import *

## 1. Definición de un autómata

`syms` crea variables cuyo valor es su propio nombre. La tabla asocia a cada símbolo una etiqueta LaTeX
que se usa en los dibujos. Los eventos no observables se dibujan con líneas punteadas.

In [ ]:
syms('q1 q2 q3 a1 b1 e')
table = [(a1, '\\alpha_1'), (b1, '\\beta_1'), (q1, 'q_1'), (q2, 'q_2'), (q3, 'q_3')]
X = [q1, q2, q3]
E = [a1, b1, e]
X0 = [q1]
Xm = [q1, q3]
T = [(q1, b1, q2), (q2, b1, q3), (q2, e, q3), (q3, a1, q1), (q2, a1, q2)]
G1 = fsa(X, E, T, X0, Xm, table, Sigobs=[a1, b1], name='$G_1$ -- maquina')
draw(G1)

Un autómata en la última línea de una celda también se dibuja automáticamente:

In [ ]:
G1

In [ ]:
print(G1)
print('Estados:', G1.X, ' Eventos:', G1.Sigma)
print('Gamma(q2) =', G1.Gamma(q2), '   delta(q1, b1) =', G1.delta(q1, b1))
print('¿Es determinista?', G1.is_dfa())

## 2. Operaciones

| Operación | Notación |
|---|---|
| Producto | `G1 & G2` |
| Composición paralela | `G1 // G2` |
| Unión de lenguajes | `G1 + G2` |
| Concatenación | `G1 * G2` |
| Complemento | `~G1` |
| Diferencia | `G1 - G2` |
| Cociente | `G1 / G2` |
| Clausura de Kleene | `kleeneclos(G1)` |
| Parte accesible, coaccesible, trim | `ac(G)`, `coac(G)`, `trim(G)` |

In [ ]:
syms('a b c d e f s1 s2 s3 s4')
tab = [(a, '\\alpha'), (b, '\\beta'), (s1, 'x_1'), (s2, 'x_2'), (s3, 'x_3'), (s4, 'x_4')]
H1 = fsa([s1, s2, s3, s4], [a, b, c], [(s1, a, s2), (s2, b, s3), (s3, c, s4), (s1, b, s4)],
         [s1], [s2, s3, s4], tab, name='$H_1$')
H2 = fsa([s1, s2, s3, s4], [a, b, c, d, e, f],
         [(s1, a, s2), (s2, e, s3), (s3, f, s4), (s1, b, s3), (s1, f, s4), (s2, f, s4)],
         [s1], [s1, s2], tab, name='$H_2$')
draw(H1, H2, (H1 // H2).setpar(name='$H_1 \\parallel H_2$'))

In [ ]:
draw((H1 + H2).setpar(name='$L_m(H_1) \\cup L_m(H_2)$'),
     (~H1).setpar(name='$L_m(H_1)^c$'))

## 3. Observador y proyección

`observer(G, Sigma_o)` calcula el autómata determinista que genera y marca la proyección de los lenguajes
de `G` sobre los eventos observables `Sigma_o` (por defecto `G.Sigobs`).

In [ ]:
syms('a b c d f u v g')
X= [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
E = [a, b, c, d, f, u, v, g]
T = [(1, f, 2), (1, a, 4), (2, u, 3), (2, a, 5), (3, c, 7), (4, f, 5),
     (4, b, 8), (5, u, 6), (5, b, 9), (6, g, 7), (7, v, 5), (7, b, 10),
     (8, d, 1), (8, f, 9), (9, u, 10), (9, d, 2), (10, d, 3), (8, v, 12),
     (12, f, 11), (11, d, 1)]

G = fsa(X, E , T, [1], [1], Sigobs=[a, b, c, d, g], name='$G$')
draw(G, observer(G).setpar(name='$Obs(G)$'))

## 4. Diagnóstico de fallas

El diagnosticador es el observador de $G \| A_l$, donde $A_l$ es el autómata rotulador de la falla
(`N`: comportamiento normal, `Y`: ocurrió la falla).

In [ ]:
#syms('a b c d e f')
#X = [1, 2, 3, 4, 5, 6, 7]
#E =  [a, b, c, d, e, f]
#Eobs = [a, b, c, d, e]
#T = [(1, c, 2), (1, a, 5), (2, f, 3), (3, e, 4), (4, d, 4),
#    (5, b, 6), (6, d, 6), (7, e, 7), (3, a, 7)]
#G = fsa(X, E, T, [1], Sigobs=Eobs, name='$G$')
draw(G, diagnoser(G, f))

# Algoritmo de verificación de diagnosticabilidad de  Moreira

In [ ]:
# Step 1.1
Euo = G.Sigma - G.Sigobs
G_N1 = G.deletevent(f)
draw(G_N1.setpar(name='paso 1.1: $G_{N1}$'))



In [ ]:
# Step 1.2
GN = ac(G_N1)
draw(GN.setpar(name='paso 1.2: $GN$'))

In [ ]:
# Step 2.1
Al = labelfailaut(f)
draw(Al.setpar(name='paso 2.1: $A_L$'))


In [ ]:

# Step 2.2
Gl = G // Al
Xml = [(x, l) for (x, l) in Gl.X if l == 'Y']
Gl = Gl.setpar(Xm=Xml)
draw(Gl.setpar(name='paso 2.2: $G_L$'))

In [ ]:
# Step 2.3
GY = coac(Gl)
draw(GY.setpar(name='paso 2.3: $G_Y$'))


In [ ]:
# Step 3
R = [(event, event + '_R') for event in Euo]
GN = renamevents(GN, R)
draw(GN.setpar(name='paso 2.3: $G_Y$'))



In [ ]:
#step 4
GV = GN // GY
XmV = [(xN, (xY, l)) for (xN, (xY, l)) in GV.X if l == 'Y']
GV = GV.setpar(Xm=XmV)
draw(GV.setpar(name='paso 4: $G_V'))




In [ ]:
# We need to find the nontrivial SCCs
SCC = strconncomps(GV.Graph)
C_1 = selfloopnodes(GV.Graph)
SCC_NT = [C for C in SCC if (len(C) > 1) | (C in C_1)]
# we are searching for a counter-example
diagnosable = True
for C in SCC_NT:
    for x_V in (set(C) & GV.Xm):
        if (GV.Gamma(x_V) & G.Sigma) != set([]):
            diagnosable = False
            print(C)


print(diagnosable)

## Función completa

In [ ]:
def verifydiagnosability(G, f):
    Euo = G.Sigma - G.Sigobs  # obtaining Sigma_uo
    # Step 1.1
    G_N1 = G.deletevent(f)
    # Step 1.2
    GN = ac(G_N1)
    # Step 2.1
    Al = labelfailaut(f)
    # Step 2.2
    Gl = G // Al
    Xml = [(x, l) for (x, l) in Gl.X if l == 'Y']
    Gl = Gl.setpar(Xm=Xml)
    # Step 2.3
    GY = coac(Gl)
    # Step 3
    R = [(event, event + '_R') for event in Euo]
    GN = renamevents(GN, R)
    # Step 4
    GV = GN // GY
    XmV = [(xN, (xY, l)) for (xN, (xY, l)) in GV.X if l == 'Y']
    GV = GV.setpar(Xm=XmV)
    # Step 5
    # We need to find the nontrivial SCCs
    SCC = strconncomps(GV.Graph)
    C_1 = selfloopnodes(GV.Graph)
    SCC_NT = [C for C in SCC if (len(C) > 1) | (C in C_1)]
    # we are searching for a counter-example
    diagnosable = True
    for C in SCC_NT:
        for x_V in (set(C) & GV.Xm):
            if (GV.Gamma(x_V) & G.Sigma) != set([]):
                diagnosable = False
                break
    return diagnosable

### funcion unica

In [ ]:
verifydiagnosability(G, f)
